# SNP pipelines (P1_SNP: Lasso, P2_SNP: RFE + Random Forest)

Inputs: `all_data_only` (samples x SNPs, indexed by sample ID) and `y` (list of 0/1 labels in the same row order).
Each pipeline writes out-of-fold test predictions to `pred_probs/` for `03_stacking.ipynb`.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import Lasso
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, balanced_accuracy_score, precision_score, recall_score
from imblearn.metrics import specificity_score
import shap
from sklearn.feature_selection import RFE
os.makedirs('pred_probs', exist_ok=True)
os.makedirs('imp_snps', exist_ok=True)
os.makedirs('splits', exist_ok=True)
os.makedirs('results', exist_ok=True)

In [ ]:
def only_lasso(x_train_s, y_train_s, x_test_s, tol, my_alpha = 0.02, my_thresh=0.5):
    lasso_model = Lasso(alpha=my_alpha, random_state=9, tol=tol)
    lasso_model.fit(x_train_s.astype('int'), y_train_s)

    preds_proba = lasso_model.predict(x_test_s)
    
    binary_preds = (preds_proba>=my_thresh).astype('int')
    
    return binary_preds, preds_proba

In [ ]:
def pred_cv(x, y, tol, my_alpha, flex=0, out_name='snps_rfe_rf', rf_params=None, rfe_params=None):
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    X = x.reset_index()
    id_col = X.columns[0]

    y_true =[]
    y_pred = []
    y_probs = []

    c=0

    for train_ix, test_ix in cv.split(X, y):

        # split data
        X_train, X_test = X[X.index.isin(train_ix)], X[X.index.isin(test_ix)]
        y_train, y_test = pd.DataFrame([y[i] for i in train_ix], columns=['OUTCOME']), \
                                pd.DataFrame([y[i] for i in test_ix], columns=['OUTCOME'])
        
        y_train['index'] = X_train[id_col].values
        y_test['index'] = X_test[id_col].values
        
        X_train.set_index(id_col, inplace=True)
        X_test.set_index(id_col, inplace=True)
        y_train.set_index('index', inplace=True)
        y_test.set_index('index', inplace=True)
        
        pd.DataFrame(test_ix).to_csv('splits/test_f{}.csv'.format(c))

        data = pd.concat([X_train, y_train], axis=1)

        x_train_s, y_train_s = data.iloc[:, :-1], data.iloc[:, -1]

        if flex == 0:
            if rf_params is None:
                rf_params = {"max_features": 45, "min_samples_split": 30, "min_samples_leaf": 15, "n_estimators": 400}

            model = RandomForestClassifier(**rf_params,
        bootstrap=False,
         random_state = 42,
                n_jobs=-1
            )

            if rfe_params is None:
                rfe_params = {"n_features_to_select": 30, "step": 100}

            selector = RFE(estimator=model, **rfe_params)
    
            selector = selector.fit(x_train_s, y_train_s)
        
            selected_features =x_train_s.columns[selector.support_]

            x_train_s = x_train_s.loc[:, selected_features]
            X_test = X_test.loc[:, selected_features]

            print(x_train_s.shape)

            model.fit(x_train_s, y_train_s)

            # evaluate model
            yhat = model.predict(X_test)
            y_prob = model.predict_proba(X_test)

            pd.DataFrame(y_prob, index=list(y_test.index)).to_csv(f'pred_probs/{out_name}_split{c}.csv', header=None)

            y_true.append(y_test)
            y_pred.append(yhat)
            y_probs.append(np.round(y_prob[:, 1],5))

            explainer = shap.TreeExplainer(model)
            shap_exp_vals = explainer.shap_values(x_train_s)
            shap_pos = shap_exp_vals[1] if isinstance(shap_exp_vals, list) else shap_exp_vals[:, :, 1]
            sorted_vals = np.abs(shap_pos).mean(0)
            
            feature_imp =pd.DataFrame(list(zip(x_train_s.columns, sorted_vals)), columns =['colname', 'shap_values'])
            feature_imp.sort_values(by=['shap_values'], ascending=False, inplace=True)
            feature_imp.to_csv("imp_snps/shap_{}_split{}.csv".format(out_name, c))

        elif flex==1:
            binary_preds, preds_proba = only_lasso(x_train_s, y_train_s, X_test, tol, my_alpha)

            pd.DataFrame(preds_proba, index=list(y_test.index)).to_csv(f'pred_probs/{out_name}_split{c}.csv', header=None)
        
            y_true.append(y_test)
            y_pred.append(binary_preds)
            y_probs.append(np.round(preds_proba,5))

        c=c+1

    return y_true, y_pred, y_probs

## P1_SNP: Lasso

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    y_true, y_pred, y_probs = pred_cv(all_data_only, y, 0.00000001, my_alpha=0.01, flex=1, out_name='snps_lasso')

In [ ]:
roc_auc = []
pr_auc = []
precs = []
recs = []
specifs = []
balanced_accs =[]
tprs = []

mean_fpr = np.linspace(0, 1, 10)

for i, j, k in zip(y_true, y_pred, y_probs):
    roc_auc.append(roc_auc_score(i, k, average='weighted'))
    precs.append(precision_score(i, j, average='weighted'))
    recs.append(recall_score(i, j, average='weighted'))
    precision_pr, recall_pr, _ = precision_recall_curve(i, k)
    specifs.append(specificity_score(i, j, average='weighted'))
    
    balanced_accs.append(balanced_accuracy_score(i, j))

    fpr, tpr, _ = roc_curve(i, k)
    tprs.append(np.interp(mean_fpr, fpr, tpr))

In [ ]:
print(roc_auc)
print("roc_auc: ", np.mean(roc_auc), np.std(roc_auc))

print(precs)
print("precision_score: ", np.mean(precs), np.std(precs))

print(recs)
print("recall_score: ", np.mean(recs), np.std(recs))

print(specifs)
print("specificity: ", np.mean(specifs), np.std(specifs))

print(balanced_accs)
print("balanced_auc: ", np.mean(balanced_accs), np.std(balanced_accs))

In [ ]:
temp__ = pd.DataFrame({
    "roc_auc": np.round(roc_auc, decimals=4),
    "precision_score" : np.round(precs, decimals=4) ,
    "recall_score" : np.round(recs, decimals=4),
    "specificity": np.round(specifs, decimals=4),
    "balanced_auc": np.round(balanced_accs, decimals=4),
})

temp__.to_csv('results/p1_snp.csv')

## P2_SNP: hyperparameter selection on the 10 folds (setting best in the most folds)

In [ ]:
import warnings

In [ ]:
param_grid = [
    {"my_alpha": 0.01, "rf_params": {"max_features": 45, "min_samples_split": 30, "min_samples_leaf": 15, "n_estimators": 400}, "rfe_params": {"n_features_to_select": 30, "step": 100}},
    {"my_alpha": 0.01, "rf_params": {"max_features": 45, "min_samples_split": 30, "min_samples_leaf": 15, "n_estimators": 400}, "rfe_params": {"n_features_to_select": 120, "step": 450}},
    {"my_alpha": 0.01, "rf_params": {"max_features": 45, "min_samples_split": 30, "min_samples_leaf": 15, "n_estimators": 400}, "rfe_params": {"n_features_to_select": 150, "step": 450}},
    {"my_alpha": 0.01, "rf_params": {"max_features": 45, "min_samples_split": 20, "min_samples_leaf": 11, "n_estimators": 400}, "rfe_params": {"n_features_to_select": 150, "step": 450}},
    {"my_alpha": 0.01, "rf_params": {"max_features": 25, "min_samples_split": 30, "min_samples_leaf": 17, "n_estimators": 400, "class_weight": "balanced_subsample"}, "rfe_params": {"n_features_to_select": 70, "step": 420}},
]

fold_aucs = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for params in param_grid:
        y_true, y_pred, y_probs = pred_cv(all_data_only, y, 0.00000001, flex=0, out_name='tuning', **params)
        fold_aucs.append([roc_auc_score(i, k) for i, k in zip(y_true, y_probs)])

fold_aucs = np.array(fold_aucs)
fold_wins = np.bincount(fold_aucs.argmax(axis=0), minlength=len(param_grid))
best = max(range(len(param_grid)), key=lambda p: (fold_wins[p], fold_aucs[p].mean()))

print(pd.DataFrame({"fold_wins": fold_wins, "mean_auc": fold_aucs.mean(axis=1).round(4)}))
best_params = param_grid[best]
print(best_params)

## P2_SNP: RFE + Random Forest

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    y_true, y_pred, y_probs = pred_cv(all_data_only, y, 0.00000001, my_alpha=best_params['my_alpha'], flex=0, out_name='snps_rfe_rf', rf_params=best_params['rf_params'], rfe_params=best_params['rfe_params'])

In [ ]:
roc_auc = []
pr_auc = []
precs = []
recs = []
specifs = []
balanced_accs =[]
tprs = []

mean_fpr = np.linspace(0, 1, 10)

for i, j, k in zip(y_true, y_pred, y_probs):
    roc_auc.append(roc_auc_score(i, k, average='weighted'))
    precs.append(precision_score(i, j, average='weighted'))
    recs.append(recall_score(i, j, average='weighted'))
    precision_pr, recall_pr, _ = precision_recall_curve(i, k)
    specifs.append(specificity_score(i, j, average='weighted'))
    
    balanced_accs.append(balanced_accuracy_score(i, j))

    fpr, tpr, _ = roc_curve(i, k)
    tprs.append(np.interp(mean_fpr, fpr, tpr))

In [ ]:
print(roc_auc)
print("roc_auc: ", np.mean(roc_auc), np.std(roc_auc))

print(precs)
print("precision_score: ", np.mean(precs), np.std(precs))

print(recs)
print("recall_score: ", np.mean(recs), np.std(recs))

print(specifs)
print("specificity: ", np.mean(specifs), np.std(specifs))

print(balanced_accs)
print("balanced_auc: ", np.mean(balanced_accs), np.std(balanced_accs))

In [ ]:
temp__ = pd.DataFrame({
    "roc_auc": np.round(roc_auc, decimals=4),
    "precision_score" : np.round(precs, decimals=4) ,
    "recall_score" : np.round(recs, decimals=4),
    "specificity": np.round(specifs, decimals=4),
    "balanced_auc": np.round(balanced_accs, decimals=4),
})

temp__.to_csv('results/p2_snp.csv')

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(15, 6))

axes = axes.flatten()

for i, ax in enumerate(axes):
    temp = np.asarray(y_probs[i])
    labels_i = np.asarray(y_true[i]).ravel()

    positive = temp[labels_i == 1]
    negative = temp[labels_i == 0]

    ax.hist(positive, color='red', label='Cases')
    ax.hist(negative, color='blue', label='Controls')
    ax.set_title(f'Fold {i+1} (n={len(y_true[i])})')

fig.supxlabel('Prediction Probabilities')
fig.supylabel('Number of Individuals')
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc='lower right')

plt.tight_layout()
    
plt.show()